In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages,prompt)
    add_assistant_message(messages,"```json")
    text=chat(messages,stop_sequences=["```"])
    return json.loads(text)

In [7]:
datasets = generate_dataset()
# datasets

with open("datasets.json","w") as f:
    json.dump(datasets,f,indent=2)

Running The Eval

In [17]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [26]:
def grade_by_model(test_case, output):
    # Create evaluation prompt using the actual task and model output.
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case['task']}
Solution:
{output}

Respond with a JSON object only, using this schema:
{{
  \"strengths\": [\"...\"],
  \"weaknesses\": [\"...\"],
  \"reasoning\": \"...\",
  \"score\": 1
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, temperature=0)
    eval_text = eval_text.strip()

    if eval_text.startswith("```"):
        lines = eval_text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        eval_text = "\n".join(lines).strip()

    try:
        model_grade = json.loads(eval_text)
    except json.JSONDecodeError:
        start = eval_text.find("{")
        end = eval_text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"Could not parse grade JSON: {eval_text}")
        model_grade = json.loads(eval_text[start:end + 1])

    if "score" not in model_grade:
        raise ValueError(f"Grader response missing 'score': {model_grade}")

    return {
        "score": model_grade["score"],
        "reasoning": model_grade.get("reasoning", ""),
        "strengths": model_grade.get("strengths", []),
        "weaknesses": model_grade.get("weaknesses", []),
    }


In [ ]:
# def run_test_case(test_case):
#     """Calls run_prompt, then grades the result"""
#     output = run_prompt(test_case)
    
#     # TODO - Grading
#     score = 10
    
#     return {
#         "output": output,
#         "test_case": test_case,
#         "score": score
#     }

In [27]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]
    
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

In [ ]:
# def run_eval(dataset):
#     """Loads the dataset and calls run_test_case with each case"""
#     results = []
    
#     for test_case in dataset:
#         result = run_test_case(test_case)
#         results.append(result)
    
#     return results


In [28]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [29]:
with open("datasets.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.5


In [30]:
print(json.dumps(results,indent=2))

[
  {
    "output": "# AWS Account ID Extraction from ARN\n\nHere's a comprehensive solution:\n\n```python\ndef extract_account_id_from_arn(arn):\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    \n    Args:\n        arn (str): The ARN string to parse\n        \n    Returns:\n        str: The account ID, or None if not found or if the ARN is invalid\n        \n    Examples:\n        >>> extract_account_id_from_arn('arn:aws:iam::123456789012:user/Development/product_1/*')\n        '123456789012'\n        >>> extract_account_id_from_arn('arn:aws:s3:::my-bucket/my-key')\n        ''\n        >>> extract_account_id_from_arn('arn:aws:rds:us-east-1:123456789012:db:mysql-db')\n        '123456789012'\n    \"\"\"\n    if not isinstance(arn, str):\n        return None\n    \n    try:\n        # ARN format: arn:partition:service:region:account-id:resource\n        parts = arn.split(':')\n        \n        #